# 🔬 RAG Research Assistant — Complete Master Notebook
## RAG-Based LLM System for Intelligent Research Paper Analysis
---
### ✅ Run this ONE notebook — covers all 8 phases!
| Phase | Feature |
|---|---|
| 1 | PDF Upload & Text Extraction |
| 2 | Chunking + Embeddings + FAISS |
| 3 | Semantic Search + LLM Q&A |
| 4 | Citation Highlighting + Summary |
| 5 | Multi-PDF + Paper Comparison |
| 6 | Knowledge Graph |
| 7+8 | Multi-modal + Gradio UI |

##Why is it needed?

Researchers spend 30–40% of their time just reading papers
Keyword search (Google Scholar) finds papers but doesn't understand them
ChatGPT answers from training data, not your specific uploaded paper
No existing free tool does Q&A + citations + comparison + knowledge graph together
Students doing literature reviews waste weeks manually reading 50+ papers

##How it works (core logic)

1. PDF gets broken into small text chunks (512 tokens each)
2. Each chunk is converted into a number vector (embedding) that captures meaning

3. These vectors are stored in FAISS — a fast similarity search database

4. When you ask a question, your question also becomes a vector

5. FAISS finds the 5 most similar chunks to your question

6. Those chunks + your question are sent to Claude/GPT-4 as context

7. LLM answers based only on those chunks — not from memory

8. This whole process is called RAG (Retrieval-Augmented Generation)

## ⚙️ STEP 1: Install All Dependencies (Run Once)

In [1]:
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q groq
!pip install -q pymupdf
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q networkx pyvis
!pip install -q gradio
!pip install -q Pillow
print('✅ ALL packages installed successfully!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

## 🔑 STEP 2: Configuration — Set Your API Key & Paths

In [2]:
import os, json, pickle, re, shutil, io
import numpy as np
import fitz  # PyMuPDF
from PIL import Image
from collections import defaultdict

# ── YOUR GROQ API KEY ─────────────────────────────────
#GROQ_API_KEY = ''  # ← PASTE YOUR KEY HERE
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
# ──────────────────────────────────────────────────────

# Project folders — all local, no Drive needed
BASE_DIR    = '/content/RAG_Project'
PDF_DIR     = f'{BASE_DIR}/uploaded_pdfs'
TEXT_DIR    = f'{BASE_DIR}/extracted_texts'
INDEX_DIR   = f'{BASE_DIR}/faiss_index'
OUTPUT_DIR  = f'{BASE_DIR}/outputs'
FIGURES_DIR = f'{BASE_DIR}/figures'

for d in [BASE_DIR, PDF_DIR, TEXT_DIR, INDEX_DIR, OUTPUT_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# Chunking config
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
LLM_MODEL     = 'llama-3.3-70b-versatile'

print('✅ Configuration complete!')
print(f'📁 Project folder: {BASE_DIR}')

✅ Configuration complete!
📁 Project folder: /content/RAG_Project


## 📤 STEP 3: Upload PDF Files

In [3]:
from google.colab import files

print('📤 Please select your PDF files to upload...')
uploaded = files.upload()

for filename, data in uploaded.items():
    dst = f'{PDF_DIR}/{filename}'
    with open(dst, 'wb') as f:
        f.write(data)
    print(f'✅ Uploaded: {filename}')

all_pdfs = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]
print(f'\n📚 Total PDFs ready: {len(all_pdfs)}')
for p in all_pdfs:
    print(f'  → {p}')

📤 Please select your PDF files to upload...


Saving Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper.pdf to Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper.pdf
Saving Visualizing and Understanding Convolutional Networks_RESPAPER.pdf to Visualizing and Understanding Convolutional Networks_RESPAPER.pdf
✅ Uploaded: Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper.pdf
✅ Uploaded: Visualizing and Understanding Convolutional Networks_RESPAPER.pdf

📚 Total PDFs ready: 2
  → Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper.pdf
  → Visualizing and Understanding Convolutional Networks_RESPAPER.pdf


## 📄 STEP 4: Phase 1 — Extract Text from PDFs

In [4]:
from tqdm import tqdm

def clean_text(text):
    text = re.sub(r'-(\n)', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text.strip()

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages_data = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = clean_text(page.get_text('text').strip())
        if text:
            pages_data.append({
                'page_num'  : page_num + 1,
                'text'      : text,
                'word_count': len(text.split())
            })
    doc.close()
    return pages_data

# Extract all PDFs
all_documents = {}
pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]

for pdf_file in tqdm(pdf_files, desc='Extracting PDFs'):
    pdf_path = f'{PDF_DIR}/{pdf_file}'
    doc_name = pdf_file.replace('.pdf', '')
    pages    = extract_text_from_pdf(pdf_path)
    all_documents[doc_name] = {
        'filename'   : pdf_file,
        'pages'      : pages,
        'total_pages': len(pages),
        'total_words': sum(p['word_count'] for p in pages)
    }
    # Save to JSON
    with open(f'{TEXT_DIR}/{doc_name}.json', 'w') as f:
        json.dump(all_documents[doc_name], f, indent=2)
    print(f'✅ {pdf_file}: {len(pages)} pages, {all_documents[doc_name]["total_words"]} words')

print('\n✅ Phase 1 Complete — Text extraction done!')

Extracting PDFs: 100%|██████████| 2/2 [00:00<00:00,  9.92it/s]

✅ Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper.pdf: 12 pages, 6584 words
✅ Visualizing and Understanding Convolutional Networks_RESPAPER.pdf: 16 pages, 6580 words

✅ Phase 1 Complete — Text extraction done!


## 🧩 STEP 5: Phase 2 — Chunking + Embeddings + FAISS

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size      = CHUNK_SIZE,
    chunk_overlap   = CHUNK_OVERLAP,
    separators      = ['\n\n', '\n', '. ', ' ', ''],
    length_function = len
)

all_chunks = []
chunk_id   = 0
for doc_name, doc_data in all_documents.items():
    for page in doc_data['pages']:
        for chunk in text_splitter.split_text(page['text']):
            if len(chunk.strip()) > 50:
                all_chunks.append({
                    'chunk_id': chunk_id,
                    'doc_name': doc_name,
                    'filename': doc_data['filename'],
                    'page_num': page['page_num'],
                    'text'    : chunk.strip()
                })
                chunk_id += 1

print(f'✅ Total chunks: {len(all_chunks)}')

# Embeddings
print('\n⏳ Loading embedding model...')
embedder   = SentenceTransformer(EMBEDDING_MODEL)
texts      = [c['text'] for c in all_chunks]
print(f'⏳ Generating embeddings for {len(texts)} chunks...')
embeddings = embedder.encode(texts, convert_to_numpy=True,
                              normalize_embeddings=True, show_progress_bar=True)

# FAISS index
EMBEDDING_DIM = embeddings.shape[1]
index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings.astype('float32'))

# Save
faiss.write_index(index, f'{INDEX_DIR}/research_index.faiss')
with open(f'{INDEX_DIR}/chunks_metadata.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)
config = {'embedding_model': EMBEDDING_MODEL, 'embedding_dim': EMBEDDING_DIM,
          'total_chunks': len(all_chunks), 'chunk_size': CHUNK_SIZE, 'chunk_overlap': CHUNK_OVERLAP}
with open(f'{INDEX_DIR}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'\n✅ Phase 2 Complete — FAISS index built with {index.ntotal} vectors!')

✅ Total chunks: 209

⏳ Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Generating embeddings for 209 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


✅ Phase 2 Complete — FAISS index built with 209 vectors!


## 🤖 STEP 6: Phase 3 — Semantic Search + LLM Q&A Engine

In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def semantic_search(query, top_k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_emb, k=top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            c = all_chunks[idx].copy()
            c['score'] = float(score)
            results.append(c)
    return results

def call_llm(prompt, max_tokens=1024):
    r = groq_client.chat.completions.create(
        model       = LLM_MODEL,
        messages    = [{'role': 'user', 'content': prompt}],
        temperature = 0.1,
        max_tokens  = max_tokens
    )
    return r.choices[0].message.content

def rag_answer(query, top_k=5):
    chunks  = semantic_search(query, top_k=top_k)
    context = '\n\n'.join([
        f'[Source {i+1}: {c["doc_name"]}, Page {c["page_num"]}]\n{c["text"]}'
        for i, c in enumerate(chunks)
    ])
    prompt = f"""You are an expert research assistant. Answer using ONLY the context below.
Always cite [Source N] in your answer.
If not found say: 'Not found in the provided documents.'

CONTEXT:
{context}

QUESTION: {query}
ANSWER:"""
    answer    = call_llm(prompt)
    citations = '\n'.join([
        f'[Source {i+1}] {c["doc_name"]} | Page {c["page_num"]} | Score: {c["score"]:.3f}'
        for i, c in enumerate(chunks)
    ])
    return answer, citations

# Test
test_ans, test_cit = rag_answer('What is the main contribution of this paper?')
print('🤖 TEST ANSWER:')
print(test_ans)
print('\n📌 CITATIONS:')
print(test_cit)
print('\n✅ Phase 3 Complete — RAG Q&A engine ready!')

🤖 TEST ANSWER:
Not found in the provided documents. [Source 1], [Source 2], [Source 3], [Source 4], and [Source 5] do not mention the main contribution of the paper. [Source N]

📌 CITATIONS:
[Source 1] Visualizing and Understanding Convolutional Networks_RESPAPER | Page 10 | Score: 0.297
[Source 2] Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper | Page 11 | Score: 0.254
[Source 3] Visualizing and Understanding Convolutional Networks_RESPAPER | Page 10 | Score: 0.247
[Source 4] Visualizing and Understanding Convolutional Networks_RESPAPER | Page 2 | Score: 0.237
[Source 5] Visualizing and Understanding Convolutional Networks_RESPAPER | Page 11 | Score: 0.218

✅ Phase 3 Complete — RAG Q&A engine ready!


## 📝 STEP 7: Phase 4 — Paper Summary Generator

In [ ]:
def generate_paper_summary(doc_name):
    with open(f'{TEXT_DIR}/{doc_name}.json') as f:
        doc = json.load(f)
    text = ' '.join([p['text'] for p in doc['pages'][:8]])[:6000]
    prompt = f"""Summarize this research paper in structured format:

**TITLE:** ...
**AUTHORS:** ...
**ABSTRACT SUMMARY:** (2-3 sentences)
**KEY CONTRIBUTIONS:** (3 bullet points)
**METHODOLOGY:** (2 sentences)
**KEY FINDINGS:** (3 bullet points)
**KEYWORDS:** (5-7 words)

PAPER TEXT:
{text}"""
    return call_llm(prompt, max_tokens=1200)

# Generate summaries for all docs
summaries = {}
all_docs  = list(all_documents.keys())

for doc_name in all_docs:
    print(f'\n📝 Summarizing: {doc_name}...')
    summaries[doc_name] = generate_paper_summary(doc_name)
    print(summaries[doc_name])
    print('='*70)

with open(f'{OUTPUT_DIR}/summaries.json', 'w') as f:
    json.dump(summaries, f, indent=2)
print('\n✅ Phase 4 Complete — Summaries generated!')


📝 Summarizing: Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper...
**TITLE:** Going Deeper with Convolutions
**AUTHORS:** Christian Szegedy, Wei Liu, Yangqing Jia, Pierre Sermanet, Scott Reed, Dragomir Anguelov, Dumitru Erhan, Vincent Vanhoucke, Andrew Rabinovich
**ABSTRACT SUMMARY:** The authors propose a deep convolutional neural network architecture called Inception, which achieved state-of-the-art results in the ImageNet Large-Scale Visual Recognition Challenge 2014. The Inception architecture improves the utilization of computing resources, allowing for increased depth and width while keeping the computational budget constant. The authors experimentally verify the benefits of the architecture on the ILSVRC 2014 classification and detection challenges.
**KEY CONTRIBUTIONS:**
* The authors propose a new deep convolutional neural network architecture called Inception, which achieves state-of-the-art results in image classification and detection.
* The Inception architectu

## 📚 STEP 8: Phase 5 — Multi-PDF + Literature Review

In [ ]:
def compare_papers(doc1, doc2, topic):
    q_emb = embedder.encode([topic], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_emb, k=50)
    c1, c2 = [], []
    for sc, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        ch = all_chunks[idx]
        if ch['doc_name'] == doc1 and len(c1) < 3: c1.append(ch)
        if ch['doc_name'] == doc2 and len(c2) < 3: c2.append(ch)
    t1 = ' '.join([c['text'] for c in c1])[:2000]
    t2 = ' '.join([c['text'] for c in c2])[:2000]
    prompt = f"""Compare these two papers on: "{topic}"
PAPER 1 ({doc1}): {t1}
PAPER 2 ({doc2}): {t2}
Format:
**AGREEMENTS:** ...
**DIFFERENCES:** ...
**PAPER 1 UNIQUE:** ...
**PAPER 2 UNIQUE:** ...
**VERDICT:** ..."""
    return call_llm(prompt, max_tokens=1500)

def generate_literature_review(topic):
    q_emb = embedder.encode([topic], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_emb, k=30)
    by_doc = defaultdict(list)
    for sc, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        ch = all_chunks[idx]
        if len(by_doc[ch['doc_name']]) < 2:
            by_doc[ch['doc_name']].append(ch['text'])
    context = '\n\n'.join([
        f'**{doc}:** {" ".join(texts)[:1200]}'
        for doc, texts in by_doc.items()
    ])
    prompt = f"""Write an academic literature review on: "{topic}"
Using ONLY these sources:
{context}
Sections: Introduction | Overview | Key Themes | Research Gaps | Conclusion"""
    return call_llm(prompt, max_tokens=2000)

print('✅ Phase 5 Complete — Comparison & Literature Review functions ready!')
print(f'📚 Documents available: {all_docs}')
if len(all_docs) >= 2:
    print('\n💡 You can now compare papers using: compare_papers(doc1, doc2, topic)')
print('💡 Generate literature review using: generate_literature_review(topic)')

✅ Phase 5 Complete — Comparison & Literature Review functions ready!
📚 Documents available: ['Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper', 'Visualizing and Understanding Convolutional Networks_RESPAPER']

💡 You can now compare papers using: compare_papers(doc1, doc2, topic)
💡 Generate literature review using: generate_literature_review(topic)


## 🎨 STEP 9: Phase 6 — Knowledge Graph

In [ ]:
import networkx as nx
from pyvis.network import Network
from IPython.display import display, HTML

def extract_concepts(doc_name):
    with open(f'{TEXT_DIR}/{doc_name}.json') as f:
        doc = json.load(f)
    sample = ' '.join([p['text'] for p in doc['pages'][:5]])[:3000]
    prompt = f"""Extract exactly 8 key technical concepts from this text.
Return ONLY a JSON list. Example: ["neural network", "gradient descent"]
TEXT: {sample}
JSON:"""
    resp = call_llm(prompt, max_tokens=200)
    try:
        match = re.search(r'\[.*?\]', resp, re.DOTALL)
        if match: return json.loads(match.group())
    except: pass
    return []

# Build graph
doc_concepts = {}
for doc in all_docs:
    doc_concepts[doc] = extract_concepts(doc)
    print(f'📄 {doc}: {doc_concepts[doc]}')

G   = nx.Graph()
net = Network(height='600px', width='100%', bgcolor='#222222',
              font_color='white', notebook=True)

for doc in all_docs:
    G.add_node(doc, node_type='document')
    net.add_node(doc, label=doc[:20], color='#4285F4', size=25)

concept_docs = defaultdict(list)
for doc, concepts in doc_concepts.items():
    for c in concepts:
        concept_docs[c.lower()].append(doc)

for concept, docs in concept_docs.items():
    net.add_node(concept, label=concept, color='#EA4335', size=12)
    for doc in docs:
        net.add_edge(doc, concept)

graph_path = f'{OUTPUT_DIR}/knowledge_graph.html'
net.save_graph(graph_path)
display(HTML(open(graph_path).read()))
print('\n✅ Phase 6 Complete — Knowledge Graph ready!')

📄 Inception-v1 (Going Deeper With Convolutions)_Resaerchpaper: ['convolutional neural network', 'deep learning', 'Hebbian principle', 'multi-scale processing', 'object detection', 'R-CNN algorithm', 'computer vision', 'neural network architecture']
📄 Visualizing and Understanding Convolutional Networks_RESPAPER: ['Convolutional Networks', 'feature layers', 'classi er', 'ablation study', 'softmax', 'model architectures', 'Dropout', 'GPU implementations']



✅ Phase 6 Complete — Knowledge Graph ready!


## 🚀 STEP 10: Phase 7+8 — Full Gradio Web App

In [ ]:
import gradio as gr

doc_choices   = ['All Documents'] + all_docs
paper_choices = ['Select a paper'] + all_docs

def ui_answer(question, doc_filter):
    if not question.strip(): return 'Please enter a question.', ''
    chunks = semantic_search(question, top_k=5)
    if doc_filter != 'All Documents':
        chunks = [c for c in chunks if c['doc_name'] == doc_filter]
    if not chunks: return 'No relevant content found.', ''
    context = '\n\n'.join([
        f'[Source {i+1}: {c["doc_name"]}, Page {c["page_num"]}]\n{c["text"]}'
        for i, c in enumerate(chunks[:5])
    ])
    prompt  = f"""Answer using ONLY context below. Cite [Source N].
CONTEXT: {context}
QUESTION: {question}
ANSWER:"""
    answer    = call_llm(prompt)
    citations = '\n'.join([
        f'📌 [Source {i+1}] {c["doc_name"]} | Page {c["page_num"]} | Score: {c["score"]:.3f}'
        for i, c in enumerate(chunks[:5])
    ])
    return answer, citations

def ui_summary(doc_name):
    if doc_name == 'Select a paper': return 'Please select a paper.'
    return generate_paper_summary(doc_name)

def ui_compare(doc1, doc2, topic):
    if not topic.strip(): return 'Please enter a topic.'
    if len(all_docs) < 2: return 'Please upload at least 2 papers.'
    return compare_papers(doc1, doc2, topic)

def ui_lit_review(topic):
    if not topic.strip(): return 'Please enter a topic.'
    return generate_literature_review(topic)

# Build App
with gr.Blocks(theme=gr.themes.Soft(), title='RAG Research Assistant') as app:
    gr.Markdown('# 🔬 RAG-Based AI Research Assistant\n### Intelligent Research Paper Analysis & Q&A')

    with gr.Tabs():
        with gr.TabItem('💬 Q&A'):
            gr.Markdown('### Ask questions about your research papers')
            with gr.Row():
                with gr.Column(scale=2):
                    q_input  = gr.Textbox(label='Your Question', placeholder='e.g. What is the main contribution?', lines=3)
                    q_filter = gr.Dropdown(choices=doc_choices, value='All Documents', label='Filter by Document')
                    q_btn    = gr.Button('🔍 Get Answer', variant='primary')
                with gr.Column(scale=3):
                    q_answer = gr.Textbox(label='Answer', lines=10)
                    q_cite   = gr.Textbox(label='Citations', lines=5)
            q_btn.click(fn=ui_answer, inputs=[q_input, q_filter], outputs=[q_answer, q_cite])
            gr.Examples(
                examples=[['What is the main contribution?','All Documents'],
                          ['Explain the methodology','All Documents'],
                          ['What are the key findings?','All Documents']],
                inputs=[q_input, q_filter]
            )

        with gr.TabItem('📝 Summary'):
            gr.Markdown('### Auto-generate paper summary')
            s_paper = gr.Dropdown(choices=paper_choices, value='Select a paper', label='Select Paper')
            s_btn   = gr.Button('📝 Generate Summary', variant='primary')
            s_out   = gr.Textbox(label='Summary', lines=20)
            s_btn.click(fn=ui_summary, inputs=s_paper, outputs=s_out)

        with gr.TabItem('⚖️ Compare Papers'):
            gr.Markdown('### Compare two papers side by side')
            with gr.Row():
                c_doc1 = gr.Dropdown(choices=all_docs, label='Paper 1',
                                     value=all_docs[0] if all_docs else None)
                c_doc2 = gr.Dropdown(choices=all_docs, label='Paper 2',
                                     value=all_docs[1] if len(all_docs)>1 else None)
            c_topic = gr.Textbox(label='Comparison Topic', placeholder='e.g. deep learning methods')
            c_btn   = gr.Button('⚖️ Compare', variant='primary')
            c_out   = gr.Textbox(label='Comparison', lines=20)
            c_btn.click(fn=ui_compare, inputs=[c_doc1, c_doc2, c_topic], outputs=c_out)

        with gr.TabItem('📚 Literature Review'):
            gr.Markdown('### Auto-generate literature review from all papers')
            l_topic = gr.Textbox(label='Research Topic', placeholder='e.g. transformer models in NLP')
            l_btn   = gr.Button('📚 Generate Review', variant='primary')
            l_out   = gr.Textbox(label='Literature Review', lines=25)
            l_btn.click(fn=ui_lit_review, inputs=l_topic, outputs=l_out)

        with gr.TabItem('ℹ️ System Info'):
            gr.Markdown(f"""
## 📊 System Info
- **Documents:** {len(all_docs)}
- **Total Chunks:** {len(all_chunks)}
- **Embedding Model:** {EMBEDDING_MODEL}
- **LLM Model:** {LLM_MODEL}
- **Papers:** {', '.join(all_docs)}
""")

print('🚀 Launching RAG Research Assistant...')
app.launch(share=True, debug=False)

/tmp/ipykernel_619/3626925782.py:41: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='RAG Research Assistant') as app:


🚀 Launching RAG Research Assistant...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9e6d1c98031e6ab545.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 🎉 Complete System Ready!
**Title:** RAG-Based LLM System for Intelligent Research Paper Analysis and Question Answering